# 🏦 Loan Default Prediction — End-to-End ML Pipeline

**Dataset:** [Give Me Some Credit](https://www.kaggle.com/competitions/GiveMeSomeCredit/data) — Kaggle Competition  
**Records:** 150,000 real borrower records from a US financial institution  
**Tech Stack:** Python · XGBoost · SHAP · Scikit-learn · Pandas · Matplotlib  
**Goal:** Predict whether a borrower will experience serious financial distress within 2 years

> ⚠️ In lending, **false negatives are costly** — approving a loan that defaults means real financial loss.  
> That's why we optimize for **ROC-AUC** and **F1**, not just accuracy.


## 1. 📦 Imports & Setup

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb
import shap

print("✅ All libraries loaded")
print(f"XGBoost: {xgb.__version__} | SHAP: {shap.__version__}")


## 2. 📊 Dataset

**Source:** Kaggle Competition — "Give Me Some Credit"  
**Link:** https://www.kaggle.com/competitions/GiveMeSomeCredit/data

| Column | Description |
|---|---|
| `SeriousDlqin2yrs` | **Target** — 1 = defaulted (90+ day delinquency in 2 years) |
| `RevolvingUtilizationOfUnsecuredLines` | % of credit limit being used |
| `age` | Borrower age |
| `NumberOfTime30-59DaysPastDueNotWorse` | Times 30-59 days late |
| `DebtRatio` | Monthly debt / monthly income |
| `MonthlyIncome` | Monthly income (has missing values) |
| `NumberOfOpenCreditLinesAndLoans` | Open credit lines |
| `NumberOfTimes90DaysLate` | Times 90+ days late |
| `NumberRealEstateLoansOrLines` | Mortgage/real estate loans |
| `NumberOfTime60-89DaysPastDueNotWorse` | Times 60-89 days late |
| `NumberOfDependents` | Number of dependents |


In [ ]:
df = pd.read_csv('cs-training.csv', index_col=0)
df.columns = ['default', 'revolving_utilization', 'age', 'past_due_30_59',
              'debt_ratio', 'monthly_income', 'open_credit_lines',
              'times_90_days_late', 'real_estate_loans', 'past_due_60_89', 'num_dependents']

print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['default'].mean():.2%}")
print(f"\nMissing values:")
print(df.isnull().sum())
df.head()


## 3. 📈 Exploratory Data Analysis

Key questions:
- How imbalanced is the target?
- Which features separate defaulters from non-defaulters?
- Are there obvious outliers?


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🏦 Give Me Some Credit — Exploratory Data Analysis',
             color='white', fontsize=16, fontweight='bold')
for ax in axes.flatten():
    ax.set_facecolor('#161b22'); ax.spines[:].set_color('#30363d'); ax.tick_params(colors='white')

# 1. Class imbalance
ax = axes[0,0]
counts = df['default'].value_counts()
bars = ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['#2ecc71','#e74c3c'], width=0.5)
ax.set_title('Class Distribution', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', color='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
            f'{val:,} ({val/len(df):.1%})', ha='center', va='bottom', color='white', fontsize=10)

# 2. Age distribution
ax = axes[0,1]
ax.hist(df[df['default']==0]['age'], bins=40, alpha=0.7, color='#2ecc71', label='No Default')
ax.hist(df[df['default']==1]['age'], bins=40, alpha=0.7, color='#e74c3c', label='Default')
ax.set_title('Age Distribution by Default', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Age', color='white'); ax.set_ylabel('Count', color='white')
ax.legend(facecolor='#21262d', labelcolor='white')

# 3. Past due vs default rate
# NOTE: Using a local variable here, NOT adding to df
ax = axes[0,2]
_temp = df['past_due_30_59'] + df['past_due_60_89'] + df['times_90_days_late']
pdue = df.groupby(_temp.clip(0,7))['default'].mean()*100
ax.plot(pdue.index, pdue.values, 'o-', color='#e74c3c', lw=2.5, markersize=8)
ax.fill_between(pdue.index, pdue.values, alpha=0.2, color='#e74c3c')
ax.set_title('Default Rate vs Past Due Events', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Total Past Due Events', color='white'); ax.set_ylabel('Default Rate (%)', color='white')
for x, y in zip(pdue.index, pdue.values):
    ax.text(x, y+1.5, f'{y:.1f}%', ha='center', color='white', fontsize=8)

# 4. Revolving utilization
ax = axes[1,0]
ax.hist(df[df['default']==0]['revolving_utilization'].clip(0,1), bins=40, alpha=0.7, color='#3498db', label='No Default')
ax.hist(df[df['default']==1]['revolving_utilization'].clip(0,1), bins=40, alpha=0.7, color='#e74c3c', label='Default')
ax.set_title('Revolving Credit Utilization', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Utilization (capped at 1.0)', color='white'); ax.set_ylabel('Count', color='white')
ax.legend(facecolor='#21262d', labelcolor='white')

# 5. Monthly income by default
ax = axes[1,1]
data_box = [df[df['default']==0]['monthly_income'].dropna().clip(0,20000),
            df[df['default']==1]['monthly_income'].dropna().clip(0,20000)]
bp = ax.boxplot(data_box, patch_artist=True, labels=['No Default','Default'],
                boxprops=dict(facecolor='#3498db', alpha=0.7))
bp['boxes'][1].set_facecolor('#e74c3c')
ax.set_title('Monthly Income (capped $20k)', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel('Monthly Income ($)', color='white')

# 6. Correlation heatmap
ax = axes[1,2]
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', color='white', fontsize=12, fontweight='bold')
ax.tick_params(colors='white', labelsize=7)

plt.tight_layout()
plt.show()


## 4. 🧹 Data Cleaning

Real-world data is messy. We found:
- `age = 0` — impossible, data error → remove
- `revolving_utilization` max = 50,708 — should be 0–1 → cap at 99th percentile
- `debt_ratio` max = 329,664 — clearly wrong → cap at 99th percentile
- Past due columns max = 98 — sentinel value → cap at 10
- `monthly_income` has 29,731 missing values → impute with median later


In [ ]:
print("Before cleaning:", df.shape)

df = df[df['age'] > 0]

for col in ['revolving_utilization', 'debt_ratio', 'monthly_income']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)
    print(f"  Capped {col} at {cap:.2f}")

for col in ['past_due_30_59', 'times_90_days_late', 'past_due_60_89']:
    df[col] = df[col].clip(upper=10)

print(f"\nAfter cleaning: {df.shape}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print("(Will be handled by SimpleImputer in pipeline)")


## 5. 🔧 Feature Engineering

We create **5 new features** from domain knowledge about credit risk:

| New Feature | Formula | Why it matters |
|---|---|---|
| `total_past_due` | sum of all 3 past-due columns | Overall delinquency history |
| `monthly_debt_payment` | debt_ratio × monthly_income | Actual $ owed per month |
| `income_per_dependent` | income / (dependents + 1) | Disposable income per person |
| `is_ever_late` | total_past_due > 0 | Binary: any history of lateness |
| `high_utilization` | revolving_utilization > 0.75 | Credit stress flag |


In [ ]:
df['total_past_due']       = df['past_due_30_59'] + df['past_due_60_89'] + df['times_90_days_late']
df['monthly_debt_payment'] = df['debt_ratio'] * df['monthly_income'].fillna(df['monthly_income'].median())
df['income_per_dependent'] = df['monthly_income'] / (df['num_dependents'].fillna(0) + 1)
df['is_ever_late']         = (df['total_past_due'] > 0).astype(int)
df['high_utilization']     = (df['revolving_utilization'] > 0.75).astype(int)

print(f"Features after engineering: {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")


## 6. ✂️ Train/Test Split + Preprocessing

**Stratified split** preserves the 6.68% default rate in both sets.

**Preprocessing:**
1. `SimpleImputer(median)` — fills NaNs
2. `StandardScaler` — normalizes features to mean=0, std=1


In [ ]:
features = [c for c in df.columns if c != 'default']
X = df[features]
y = df['default']

print(f"Features ({len(features)}): {features}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Default rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}")

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_sc = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_sc  = scaler.transform(imputer.transform(X_test))

scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
print(f"\nClass imbalance ratio: {scale_pos_weight:.1f}:1")
print("✅ Preprocessing done.")


## 7. 🚀 Training 4 Models

We use `class_weight='balanced'` and `scale_pos_weight` instead of SMOTE — same mathematical effect, much faster on 120K rows.

For each model we find the **optimal classification threshold** using Precision-Recall curve.


In [ ]:
def find_optimal_threshold(y_test, y_prob):
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    return thresholds[np.argmax(f1_scores[:-1])]

models_config = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000,
                                               class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                   min_samples_leaf=10, class_weight='balanced',
                                                   n_jobs=-1, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                       max_depth=5, subsample=0.8, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                              subsample=0.8, colsample_bytree=0.8,
                                              scale_pos_weight=scale_pos_weight,
                                              eval_metric='logloss', random_state=42,
                                              verbosity=0, n_jobs=-1)
}

results = {}
print("🚀 Training models...\n")
for name, model in models_config.items():
    model.fit(X_train_sc, y_train)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    thresh = find_optimal_threshold(y_test, y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    results[name] = {
        'model': model, 'y_prob': y_prob, 'y_pred': y_pred, 'threshold': thresh,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'f1':      f1_score(y_test, y_pred),
        'pr_auc':  average_precision_score(y_test, y_prob),
    }
    print(f"  ✅ {name:25s}: ROC-AUC={results[name]['roc_auc']:.4f} | "
          f"F1={results[name]['f1']:.4f} | PR-AUC={results[name]['pr_auc']:.4f} | "
          f"Threshold={thresh:.3f}")


## 8. 📊 Model Comparison Dashboard

In [ ]:
model_names = ['XGBoost', 'Gradient Boosting', 'Random Forest', 'Logistic Regression']
colors_map = {'XGBoost':'#e74c3c', 'Gradient Boosting':'#f39c12',
              'Random Forest':'#2ecc71', 'Logistic Regression':'#3498db'}
bar_colors = [colors_map[m] for m in model_names]

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#0d1117')
G = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

def style_ax(ax):
    ax.set_facecolor('#161b22'); ax.spines[:].set_color('#30363d'); ax.tick_params(colors='white')

ax1 = fig.add_subplot(G[0,0]); style_ax(ax1)
aucs = [results[m]['roc_auc'] for m in model_names]
bars = ax1.bar(model_names, aucs, color=bar_colors, width=0.5)
ax1.set_ylim(0.83, 0.89)
ax1.set_title('ROC-AUC Score', color='white', fontsize=13, fontweight='bold')
ax1.set_xticklabels([m.replace(' ','\n') for m in model_names], color='white', fontsize=9)
for bar, val in zip(bars, aucs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
             f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')

ax2 = fig.add_subplot(G[0,1]); style_ax(ax2)
f1s = [results[m]['f1'] for m in model_names]
bars2 = ax2.bar(model_names, f1s, color=bar_colors, width=0.5)
ax2.set_ylim(0.38, 0.50)
ax2.set_title('F1 Score (Threshold Optimized)', color='white', fontsize=13, fontweight='bold')
ax2.set_xticklabels([m.replace(' ','\n') for m in model_names], color='white', fontsize=9)
for bar, val in zip(bars2, f1s):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
             f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')

ax3 = fig.add_subplot(G[0,2]); style_ax(ax3)
ax3.plot([0,1],[0,1],'w--', alpha=0.3, lw=1)
for name in model_names:
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    ax3.plot(fpr, tpr, label=f"{name} ({results[name]['roc_auc']:.3f})", color=colors_map[name], lw=2)
ax3.set_title('ROC Curves', color='white', fontsize=13, fontweight='bold')
ax3.set_xlabel('False Positive Rate', color='white', fontsize=9)
ax3.set_ylabel('True Positive Rate', color='white', fontsize=9)
ax3.legend(fontsize=7.5, facecolor='#21262d', labelcolor='white', loc='lower right')

ax4 = fig.add_subplot(G[1,0]); style_ax(ax4)
for name in model_names:
    p, r, _ = precision_recall_curve(y_test, results[name]['y_prob'])
    ax4.plot(r, p, label=f"{name} (AP={results[name]['pr_auc']:.3f})", color=colors_map[name], lw=2)
ax4.set_title('Precision-Recall Curves', color='white', fontsize=13, fontweight='bold')
ax4.set_xlabel('Recall', color='white', fontsize=9)
ax4.set_ylabel('Precision', color='white', fontsize=9)
ax4.legend(fontsize=7.5, facecolor='#21262d', labelcolor='white')

best_name = max(results, key=lambda m: results[m]['roc_auc'])
ax5 = fig.add_subplot(G[1,1]); style_ax(ax5)
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
            xticklabels=['No Default','Default'], yticklabels=['No Default','Default'],
            annot_kws={'size': 14, 'weight': 'bold'})
ax5.set_title(f'Confusion Matrix\n{best_name}', color='white', fontsize=12, fontweight='bold')
ax5.tick_params(colors='white')

ax6 = fig.add_subplot(G[1,2]); style_ax(ax6)
pr_aucs = [results[m]['pr_auc'] for m in model_names]
bars3 = ax6.bar(model_names, pr_aucs, color=bar_colors, width=0.5)
ax6.set_ylim(0.35, 0.43)
ax6.set_title('PR-AUC Score', color='white', fontsize=13, fontweight='bold')
ax6.set_xticklabels([m.replace(' ','\n') for m in model_names], color='white', fontsize=9)
for bar, val in zip(bars3, pr_aucs):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
             f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')

fig.suptitle('🏦 Give Me Some Credit — Model Comparison Dashboard',
             color='white', fontsize=16, fontweight='bold', y=1.01)
plt.show()
print(f"\n🏆 Best model: {best_name} (ROC-AUC: {results[best_name]['roc_auc']:.4f})")


## 9. 📋 Classification Report

In [ ]:
best_name = max(results, key=lambda m: results[m]['roc_auc'])
print(f"Classification Report — {best_name}")
print("=" * 55)
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['No Default', 'Default']))
print(f"Optimal threshold: {results[best_name]['threshold']:.3f} (tuned for max F1)")


## 10. 🧠 SHAP Explainability

SHAP tells us **which features pushed each prediction** — essential for regulatory compliance.


In [ ]:
best_model = results['XGBoost']['model']
sample = X_test_sc[:500]

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(sample)

fig2, axes = plt.subplots(1, 2, figsize=(18, 8))
fig2.patch.set_facecolor('#0d1117')

ax = axes[0]; ax.set_facecolor('#161b22'); ax.spines[:].set_color('#30363d'); ax.tick_params(colors='white')
mean_abs = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs)[-12:]
colors_s = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, 12))
bars_s = ax.barh(range(12), mean_abs[top_idx], color=colors_s)
ax.set_yticks(range(12)); ax.set_yticklabels([features[i] for i in top_idx], color='white', fontsize=9)
ax.set_title('Top 12 Features by SHAP Importance\n(XGBoost)', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|', color='white', fontsize=10)
for bar, val in zip(bars_s, mean_abs[top_idx]):
    ax.text(val+0.0002, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', color='white', fontsize=8)

ax2 = axes[1]; ax2.set_facecolor('#161b22'); ax2.spines[:].set_color('#30363d'); ax2.tick_params(colors='white')
for i, feat_idx in enumerate(top_idx[::-1][-10:]):
    sv = shap_values[:, feat_idx]
    fv = sample[:, feat_idx]
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-8)
    sc = ax2.scatter(sv, [i]*len(sv), c=fv_norm, cmap='coolwarm', alpha=0.4, s=10)
ax2.set_yticks(range(10))
ax2.set_yticklabels([features[i] for i in top_idx[::-1][-10:]], color='white', fontsize=9)
ax2.axvline(0, color='white', linestyle='--', alpha=0.4, lw=1)
ax2.set_title('SHAP Value Distribution', color='white', fontsize=13, fontweight='bold')
ax2.set_xlabel('SHAP Value (positive = increases default risk)', color='white', fontsize=10)
plt.colorbar(sc, ax=ax2, label='Feature Value (normalized)').ax.yaxis.label.set_color('white')

fig2.suptitle('🧠 SHAP Explainability — Why Does a Borrower Default?',
              color='white', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 11. 💰 Business Impact Analysis

In [ ]:
avg_loan = 12000
avg_loss_rate = 0.65

best_name = max(results, key=lambda m: results[m]['roc_auc'])
y_pred = results[best_name]['y_pred']

tp = ((y_pred==1) & (y_test==1)).sum()
fn = ((y_pred==0) & (y_test==1)).sum()
fp = ((y_pred==1) & (y_test==0)).sum()

loss_prevented  = tp * avg_loan * avg_loss_rate
missed_defaults = fn * avg_loan * avg_loss_rate
lost_business   = fp * avg_loan * 0.03
net_benefit     = loss_prevented - missed_defaults - lost_business

print("=" * 55)
print("        BUSINESS IMPACT ANALYSIS (Test Set)")
print("=" * 55)
print(f"Best model:              {best_name}")
print(f"Test set borrowers:      {len(y_test):,}")
print()
print(f"✅ Defaults correctly caught:    {tp:,}")
print(f"❌ Defaults missed:              {fn:,}")
print(f"⚠️  Good loans wrongly rejected: {fp:,}")
print()
print(f"💰 Loss prevented:       ${loss_prevented:,.0f}")
print(f"💸 Loss from misses:     ${missed_defaults:,.0f}")
print(f"📉 Foregone revenue:     ${lost_business:,.0f}")
print(f"{'─'*45}")
print(f"🏆 Estimated Net Benefit: ${net_benefit:,.0f}")
print("=" * 55)


## 12. 💾 Save Model

**Important:** The `features` list must exactly match what was used during `model.fit()`.  
This notebook keeps `total_past_due_temp` out of `df` during EDA, so features list is clean.


In [ ]:
best_name = max(results, key=lambda m: results[m]['roc_auc'])

print(f"Saving: {best_name}")
print(f"Features ({len(features)}): {features}")

joblib.dump({
    'model':       results[best_name]['model'],
    'imputer':     imputer,
    'scaler':      scaler,
    'features':    features,
    'threshold':   results[best_name]['threshold'],
    'model_name':  best_name,
    'roc_auc':     results[best_name]['roc_auc'],
}, 'best_loan_model.pkl')

print(f"\n✅ Saved: best_loan_model.pkl")
print(f"   Model:     {best_name}")
print(f"   ROC-AUC:   {results[best_name]['roc_auc']:.4f}")
print(f"   Threshold: {results[best_name]['threshold']:.3f}")
print(f"   Features:  {len(features)}")


## 13. 🔮 Predict on New Borrower

In [ ]:
def predict_default(applicant: dict, bundle_path='best_loan_model.pkl') -> dict:
    bundle = joblib.load(bundle_path)
    df_app = pd.DataFrame([applicant])

    # Feature engineering — must match training exactly
    df_app['total_past_due']       = df_app['past_due_30_59'] + df_app['past_due_60_89'] + df_app['times_90_days_late']
    df_app['monthly_debt_payment'] = df_app['debt_ratio'] * df_app['monthly_income']
    df_app['income_per_dependent'] = df_app['monthly_income'] / (df_app['num_dependents'] + 1)
    df_app['is_ever_late']         = int(df_app['total_past_due'].values[0] > 0)
    df_app['high_utilization']     = int(df_app['revolving_utilization'].values[0] > 0.75)

    X = bundle['scaler'].transform(bundle['imputer'].transform(df_app[bundle['features']]))
    prob = float(bundle['model'].predict_proba(X)[0][1])
    pred = int(prob >= bundle['threshold'])
    risk = 'LOW' if prob < 0.2 else 'MEDIUM' if prob < 0.45 else 'HIGH' if prob < 0.7 else 'VERY HIGH'

    return {
        'default_probability': round(prob, 4),
        'prediction':          'DEFAULT' if pred else 'REPAY',
        'risk_level':          risk,
        'recommendation':      'REJECT' if pred else 'APPROVE'
    }

risky = {'revolving_utilization': 0.9, 'age': 45, 'past_due_30_59': 3,
         'debt_ratio': 0.6, 'monthly_income': 3500, 'open_credit_lines': 8,
         'times_90_days_late': 2, 'real_estate_loans': 0, 'past_due_60_89': 1, 'num_dependents': 3}

safe  = {'revolving_utilization': 0.15, 'age': 38, 'past_due_30_59': 0,
         'debt_ratio': 0.2, 'monthly_income': 8500, 'open_credit_lines': 5,
         'times_90_days_late': 0, 'real_estate_loans': 1, 'past_due_60_89': 0, 'num_dependents': 1}

print("🔴 Risky Applicant:", predict_default(risky))
print("🟢 Safe Applicant: ", predict_default(safe))


## 14. 🎯 Summary

### Model Performance (Real Kaggle Data — 150K Records)

| Model | ROC-AUC | F1 Score | PR-AUC |
|---|---|---|---|
| Gradient Boosting | **0.8694** | **0.4519** | **0.4011** |
| Random Forest | 0.8676 | 0.4444 | 0.3915 |
| XGBoost | 0.8653 | 0.4493 | 0.3942 |
| Logistic Regression | 0.8624 | 0.4394 | 0.3813 |

### Key Engineering Decisions

| Decision | Why |
|---|---|
| `class_weight='balanced'` over SMOTE | 120K rows → SMOTE too slow; same effect |
| Threshold optimization | Default 0.5 cutoff wrong for imbalanced data |
| SHAP explainability | Interpretable predictions for regulators |
| 99th percentile capping | Real-world data has extreme outliers |
| 5 engineered features | Domain knowledge about credit risk |

### Known Limitations
1. No temporal train/test split — production would use time-based split
2. Static threshold — tune per lender's risk appetite in production
3. In-memory prediction store in API — no DB persistence yet
